In [2]:
import time
import components.FL_sim
import importlib
import torch
from torch.utils.data import TensorDataset

importlib.reload(components.FL_sim)
from components.FL_sim import CustomSampler

parti=25
bs=17
size=int(100*bs*parti)
temp=torch.tensor([b
            for _ in range(size // parti)
            for b in range(parti)])
dataset=TensorDataset(temp)

dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=bs,
    sampler = CustomSampler(len(dataset), parti, False, False),
    num_workers=12, pin_memory=False, #<--------
    persistent_workers=True
)

T=[]
for b in range(100):
    t1 = time.time()
    l=0
    for b in range(parti):
        dataloader.sampler.set_agent_partition(b)
        for i, batch in enumerate(dataloader):
            l+=len(batch[0])
            assert torch.all(batch[0]==b), (f"Data mismatch at rank {b}, batch {i}: "
                                            f"\n    {batch}")
    assert l==size, (f"some data not given")
    t2 = time.time()
    T.append(t2-t1)
print(f"Average time per epoch: {sum(T)/len(T)} seconds")

Average time per epoch: 0.40958457469940185 seconds


In [2]:
import FL_sim
import importlib
import torch
from torch.utils.data import TensorDataset

importlib.reload(FL_sim)
from FL_sim import CustomSampler

parti=25
bs=17
size=int(100*bs*parti)
temp=torch.tensor([b
            for _ in range(size // parti)
            for b in range(parti)])
dataset=TensorDataset(temp)

dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=bs,
    sampler = CustomSampler(dataset, parti, False, False),
    num_workers=12, pin_memory=True, persistent_workers=True
)

for b in range(parti):
    print(b)
    dataloader.sampler.offset = b
    for i, batch in enumerate(dataloader):
        assert torch.all(batch[0]==b), (f"Data mismatch at rank {b}, batch {i}: "
                                     f"\n    {batch}")

l=0
for b in range(parti):
    print(b)
    dataloader.sampler.offset = b
    for i, batch in enumerate(dataloader):
        l+=len(batch[0])
        assert torch.all(batch[0]==b), (f"Data mismatch at rank {b}, batch {i}: "
                                     f"\n    {batch}")
assert l==size, (f"some data not given")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24


In [3]:
import FL_sim
import importlib
import torch
from torch.utils.data import TensorDataset

importlib.reload(FL_sim)
from FL_sim import CustomSampler

parti=25
bs=17
size=int(100*bs*parti)
temp=torch.tensor([b
            for _ in range(size // parti)
            for b in range(parti)])
dataset=TensorDataset(temp)

dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=bs,
    sampler = CustomSampler(dataset, parti, True, True),
    num_workers=12, pin_memory=True, persistent_workers=True
)

for b in range(parti):
    print(b)
    dataloader.sampler.offset = b
    for i, batch in enumerate(dataloader):
        assert not torch.all(batch[0]==b), (f"Data mismatch at rank {b}, batch {i}: "
                                     f"\n    {batch}")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24


In [ ]:
import gc
import torch
from torch import utils
from typing import List
from torchvision import transforms
from torchvision.datasets import SVHN, VisionDataset
import numpy as np
from torch.utils.data import Sampler, DistributedSampler
import time

from components.other_utilities.datasets import FasterSVHN


class CustomSampler(Sampler):
    def __init__(self, dataset, partitions_count, offset_schedule, shuffle_whole,
                 shuffle_in_partition, non_iid_flag=False, seed=42):
        super().__init__()

        # todo: custom sampler for non-IID data
        assert non_iid_flag is False, "Currently only IID data is supported"

        self.dataset = dataset
        self.shuffle_in_partition = shuffle_in_partition

        self.offset_schedule = offset_schedule
        self.offset_schedule_counter = 0

        self.seed = seed

        self.shuffle_whole_idx = np.arange(len(self.dataset))
        if shuffle_whole:
            g = torch.Generator()
            g.manual_seed(self.seed)
            self.shuffle_whole_idx = torch.randperm(
                len(dataset)).numpy()

        self.partitions_count = partitions_count

        self.size_of_partition = [
            len(self.shuffle_whole_idx[i::self.partitions_count])
            for i in range(self.partitions_count)]

    def check_offset(self, rank: int | str):
        # get next without advancing the iterator
        future_offset = self.offset_schedule[self.offset_schedule_counter]
        assert rank == future_offset, \
            f"the expected offset ({rank}) is different from scheduled offset ({future_offset})."

    def __iter__(self):
        offset = self.offset_schedule[self.offset_schedule_counter]

        if offset == 'ALL':
            self.offset_schedule_counter += 1
            return iter(self.shuffle_whole_idx)

        # Generate indices for the current partition
        indices = self.shuffle_whole_idx[offset::self.partitions_count]

        # shuffle the indices within the partition
        if self.shuffle_in_partition:
            in_part_idx = torch.randperm(len(self)).numpy()
            indices = indices[in_part_idx]

        self.offset_schedule_counter += 1
        return iter(indices)

    def __len__(self) -> int:
        future_offset = self.offset_schedule[self.offset_schedule_counter]
        if future_offset == 'ALL':
            return len(self.shuffle_whole_idx)
        return self.size_of_partition[future_offset]


class CustomSampler_old(Sampler):
    def __init__(self, dataset, partitions_count, shuffle_whole,
                 shuffle_in_partition, non_iid_flag=False, seed=42):
        super().__init__()

        # todo: custom sampler for non-IID data
        assert non_iid_flag is False, "Currently only IID data is supported"

        self.shuffle_in_partition = shuffle_in_partition

        self.offset = 0
        self.seed = seed
        self.dataset_len = len(dataset)

        self.shuffle_whole = shuffle_whole

        self.partitions_count = partitions_count

        shuffle_whole_idx = self.get_whole_shuffle_idx()
        self.size_of_partition = {
            i: len(shuffle_whole_idx[i::self.partitions_count])
            for i in range(self.partitions_count)}

    def get_whole_shuffle_idx(self):
        if self.shuffle_whole:
            g = torch.Generator()
            g.manual_seed(self.seed)
            return torch.randperm(self.dataset_len, generator=g).numpy()
        else:
            return np.arange(self.dataset_len)

    def __iter__(self):
        shuffle_whole_idx = self.get_whole_shuffle_idx()
        if self.offset == 'ALL':
            return iter(shuffle_whole_idx)
        # Generate indices for the current partition
        indices = shuffle_whole_idx[self.offset::self.partitions_count]

        # shuffle the indices within the partition
        if self.shuffle_in_partition:
            in_part_idx = torch.randperm(len(self)).numpy()
            indices = indices[in_part_idx]

        return iter(indices)

    def __len__(self) -> int:
        return self.size_of_partition[self.offset]


class CustomSampler_optimized(Sampler):
    def __init__(self, dataset_len, partitions_count, shuffle_whole,
                 shuffle_in_partition, non_iid_flag=False, seed=42):
        super().__init__()

        # Store only dataset length, not dataset reference
        self.dataset_len = dataset_len
        self.partitions_count = partitions_count
        self.shuffle_whole = shuffle_whole
        self.shuffle_in_partition = shuffle_in_partition
        self.seed = seed
        self.offset = 0

        # Pre-compute partition sizes to avoid recalculation
        base_size = self.dataset_len // self.partitions_count
        remainder = self.dataset_len % self.partitions_count
        self.size_of_partition = {
            i: base_size + (1 if i < remainder else 0)
            for i in range(self.partitions_count)
        }

    def get_whole_shuffle_idx(self):
        if self.shuffle_whole:
            rng = np.random.RandomState(self.seed)
            return rng.permutation(self.dataset_len)
        else:
            return np.arange(self.dataset_len)

    def __iter__(self):
        shuffle_whole_idx = self.get_whole_shuffle_idx()
        if self.offset == 'ALL':
            return iter(shuffle_whole_idx)

        # Generate indices for the current partition
        indices = shuffle_whole_idx[self.offset::self.partitions_count]

        # Shuffle within partition using numpy (faster and pickles better)
        if self.shuffle_in_partition:
            rng = np.random.RandomState(self.seed + self.offset)
            rng.shuffle(indices)

        return iter(indices)

    def __len__(self) -> int:
        if self.offset == 'ALL':
            return self.dataset_len
        return self.size_of_partition[self.offset]


class CachedDataset(utils.data.Dataset):
    def __init__(self, dataset, indexed_cache, cached_data, cached_labels):
        self.dataset = dataset
        self.cached_idx = indexed_cache
        self.cached_data = cached_data
        self.cached_labels = cached_labels

        for i in range(len(self)):
            self[i]

    def __getitem__(self, index):
        if not self.cached_idx[index]:
            temp = self.dataset[index]
            self.cached_data[index] = temp[0]
            self.cached_labels[index] = temp[1]
            self.cached_idx[index] = True
        return self.cached_data[index], self.cached_labels[index]

    def __len__(self):
        return len(self.dataset)


if __name__ == '__main__':
    dataset = [
        FasterSVHN(
            limit_count=200_000,
            root='./data/SVHN', split=s,
            transform=transforms.Compose([
                transforms.Resize(32),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.4377, 0.4438, 0.4728],
                    std=[0.1980, 0.2010, 0.1970]
                ),
            ])
        ) for s in ['train', 'test']][0]

    (pre_training_global_epochs, communication_rounds, num_agents,
        client_epochs_per_round, batch_count) = 3, 10, 5, 15, 13000

    offset_schedule: List[str | int] = []
    offset_schedule.extend(['ALL'] * pre_training_global_epochs)
    for _ in range(communication_rounds):
        offset_schedule.append('ALL')
        for ag_id in range(num_agents):
            offset_schedule.extend([ag_id] * (client_epochs_per_round + 1))
    offset_schedule.append('ALL')

    # -----------------------------------------------
    sampler_list = []
    # -----------------------------------------------
    # sampler = CustomSampler(dataset, num_agents, offset_schedule, True, True)
    # def test(dl, temp=None):
    #     dl.sampler.check_offset(temp)
    #     for b in dl:
    #         pass
    # sampler_list.append((sampler, test))
    # -----------------------------------------------
    # sampler = CustomSampler_old(dataset, num_agents, True, True)
    # def test_old(dl, temp=None):
    #     dl.sampler.offset = temp
    #     for b in dl:
    #         pass
    # sampler_list.append((sampler, test_old))
    # -----------------------------------------------
    sampler_optimized = CustomSampler_optimized(len(dataset), num_agents, True, True)
    def test_optimized(dl, temp=None):
        dl.sampler.offset = temp
        for b in dl:
            pass
    sampler_list.append((sampler_optimized, test_optimized))
    # -----------------------------------------------
    # sampler = DistributedSampler(
    #     dataset, num_replicas=num_agents, rank=0, shuffle=True,
    #     seed=42, drop_last=False)
    # def test(dl, temp=None):
    #     for b in dl:
    #         pass
    # sampler_list.append((sampler, test))
    # # # -----------------------------------------------
    # sampler = None
    # def test(dl, temp=None):
    #     length_dl = len(dl)//num_agents if temp != 'ALL' else len(dl)
    #     for i,b in enumerate(dl):
    #         if i==length_dl:
    #             break
    # sampler_list.append((sampler, test))
    # # -----------------------------------------------

    # cyclone
    for sampler, test in sampler_list:
        print(f"Testing sampler: {sampler.__class__.__name__}")

        # Optimized DataLoader settings for Windows to reduce WaitForMultipleObjects lag
        shared_train_loader = utils.data.DataLoader(
            dataset,
            batch_size=min(batch_count, 10000),
            sampler=sampler,
            num_workers=11,
            pin_memory=False,
            persistent_workers=True,
            prefetch_factor=1,
            drop_last=False,
            multiprocessing_context='spawn'
        )

        t1 = time.time()

        for _ in range(pre_training_global_epochs):
            test(shared_train_loader, 'ALL')
        for round_s in range(communication_rounds):
            test(shared_train_loader, 'ALL')
            for ag_id in range(num_agents):
                for epoch in range(client_epochs_per_round):
                    test(shared_train_loader, ag_id)
                test(shared_train_loader, ag_id)
        test(shared_train_loader, 'ALL')

        print(f"Average time per epoch: {(time.time() - t1) / 60} minutes")

        del shared_train_loader
        gc.collect()
        torch.cuda.empty_cache()
